# Text Classification using BERT

Applying BERT (Bidirectional Encoder Representations from Transformers) to the problem of multiclass text classification.
The dataset consists of text documents (news posts) categorized into multiple topics such as politics, sports, technology, and religion.

Unlike traditional models, BERT leverages deep contextual understanding of language by analyzing words based on both their left and right context.
## Workflow:
- Import Data
- Data preprocessing
- Training and validation

In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.datasets import fetch_20newsgroups
import matplotlib.pyplot as plt


import torch
import time

## 1- Import Data

In [ ]:
train_data = fetch_20newsgroups(subset='train')
test_data = fetch_20newsgroups(subset='test')

X_train = train_data.data
X_test = test_data.data

y_train = train_data.target
y_test = test_data.target

print(train_data.data[10])
print('-'*50)
print(train_data.target_names[train_data.target[10]])

In [ ]:
print(len(train_data.data))
print(len(test_data.data))
print(len(train_data.target_names))

In [ ]:
train_data.target_names

In [ ]:
pd.Series(train_data.target).map(lambda x: train_data.target_names[x]).value_counts().plot(kind='barh')

confusion matrix for later use

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):

    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(18, 14))

    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.grid(False)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=classes,
        yticklabels=classes,
        title=title,
        ylabel='True label',
        xlabel='Predicted label'
    )


    plt.setp(ax.get_xticklabels(), rotation=90, ha="right", fontsize=10)
    plt.setp(ax.get_yticklabels(), fontsize=10)

    # Annotate
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i,
                    format(cm[i, j], fmt),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if cm[i, j] > thresh else "black")

    fig.tight_layout()
    return ax

# 2. Data preprocessing
- The text must be preprocessed in a specific way for use with BERT. This is accomplished by setting preprocess_mode to ‘bert’. The BERT model and vocabulary will be automatically downloaded

- BERT can handle a maximum length of 512, but let's use less to reduce memory and improve speed.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
  return tokenizer(
      example['text'],
      padding ='max_length',
      truncation=True,
      max_length=300
  )

In [ ]:
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
train_dataset

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=20
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }


In [ ]:
args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

In [ ]:
output = trainer.predict(test_dataset)

logits = output.predictions

y_pred = np.argmax(logits, axis=1)
y_pred

In [ ]:


print("Accuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
print("\nF1 Score: {:.2f}".format(f1_score(y_test , y_pred, average='micro') * 100))

In [ ]:
print("\nF1 Score: {:.2f}".format(f1_score(y_test , y_pred, average='micro') * 100))

# Plot normalized confusion matrix
plot_confusion_matrix(y_test , y_pred, classes=test_data.target_names, normalize=True, title='Normalized confusion matrix')
plt.show()

In [ ]:
!zip -r output.zip final_model
from google.colab import files
files.download("output.zip")